### What is Skip-Gram?
#### Definition
Skip-Gram predicts surrounding context words given a target (center) word.

In simple words:
```
Target Word
      ↓
Predict
      ↓
Neighboring Words
```

#### Example
Sentence:
```
I love NLP
```

Target word:
```
love
```

Context words:
```
I

NLP
```

Training example:
```
Input:
love

Outputs:
I
NLP
```

#### Diagram
```
      love

        ↓

   ┌────┴────┐

   ↓         ↓

   I        NLP
```

### CBOW vs Skip-Gram Intuition
#### CBOW
```
Context
   ↓
Target
```

Example:
```
I      NLP

 \     /

  \   /

 Predict

  love
```

Question:
```
Given surrounding words, what is the center word?
```

#### Skip-Gram
```
Target
   ↓
Context
```

Example:
```
love
  ↓

I

NLP
```

Question:
```
Given the center word, what surrounding words are likely?
```

### Why Is It Called Skip-Gram?
Consider sentence:
```
I love NLP very much
```

Target:
```
NLP
```

Window Size:
```
2
```

Context:
```
I

love

very

much
```

The target word predicts words around it.

It effectively:
```
Skips
```

from the center word to neighboring words.

Hence:
```
Skip-Gram
```

### Creating Training Data
Sentence:
```
I love NLP very much
```

Window Size:
```
1
```

#### Target: I
Context:
```
love
```

Training Pair:
```
(I → love)
```

#### Target: love
Context:
```
I

NLP
```

Training Pairs:
```
(love → I)

(love → NLP)
```

#### Target: NLP
Context:
```
love

very
```

Training Pairs:
```
(NLP → love)

(NLP → very)
```

#### Generated Dataset
| Input | Output |
| ----- | ------ |
| I     | love   |
| love  | I      |
| love  | NLP    |
| NLP   | love   |
| NLP   | very   |
| very  | NLP    |
| very  | much   |
| much  | very   |

Notice:
```
One Target
      ↓
Many Training Examples
```

### Vocabulary Construction
Suppose vocabulary:
```
I
love
NLP
very
much
```

#### Index Mapping
| Word | Index |
| ---- | ----- |
| I    | 0     |
| love | 1     |
| NLP  | 2     |
| very | 3     |
| much | 4     |

Vocabulary Size:
```
V = 5
```

### One-Hot Encoding Input
Word:
```
love
```
↓
```
[0,1,0,0,0]
```

Word:
```
NLP
```
↓
```
[0,0,1,0,0]
```

Skip-Gram internally starts with one-hot encoded target words.


### Skip-Gram Architecture
```
Target Word
      ↓
One-Hot Vector
      ↓
Embedding Matrix
      ↓
Hidden Layer
      ↓
Output Layer
      ↓
Context Prediction
```

#### Visual Diagram
```
Target Word

     love

       ↓

 One-Hot Vector

       ↓

 Embedding Layer

       ↓

 Hidden Vector

       ↓

 Output Layer

       ↓

 Predict Context
```

Unlike CBOW:
```
One Word
      ↓
Many Predictions
```

### Embedding Matrix
Suppose:
```
Vocabulary Size = 5

Embedding Dimension = 3
```

#### Embedding Matrix
```python
W =
[
 [0.2,0.8,0.1],  # I
 [0.7,0.3,0.5],  # love
 [0.4,0.9,0.6],  # NLP
 [0.8,0.2,0.3],  # very
 [0.1,0.4,0.7]   # much
]
```

Shape:
```
5 × 3
```

Each row is a word vector.

### Embedding Lookup
Input:
```
love
```

One-Hot:
```
[0,1,0,0,0]
```

Matrix Multiplication:
```
OneHot × W
```

returns:
```
[0.7,0.3,0.5]
```

This becomes:
```
Hidden Vector
```

#### Diagram
```
love

  ↓

Embedding

  ↓

[0.7,0.3,0.5]
```

### Output Layer
Hidden vector:
```
[0.7,0.3,0.5]
```

must predict:
```
I

NLP
```

Output scores:
```
I      → 3.8

love   → 0.7

NLP    → 4.1

very   → 0.9

much   → 0.5
```

Highest scores:
```
I

NLP
```
Good prediction.


### Softmax
Scores become probabilities.

Example:
```
[
 0.42,
 0.05,
 0.45,
 0.04,
 0.04
]
```

Interpretation:
```
P(I)

=

42%
```

```
P(NLP) = 45%
```
These probabilities should be high because they are true context words.

#### Why Softmax?
Softmax converts raw scores into probabilities and ensures:
```
All Probabilities Sum To 1
```

### Training Objective
Goal:
```
Target Word
      ↓
Predict All Context Words
```

For:
```
love
```

Context:
```
I

NLP
```

The model learns:
```
P(I | love)

P(NLP | love)
```
should be high.

### Loss Function
Suppose actual context word:
```
NLP
```

Prediction:
```
0.85
```

Good prediction.
```
Small Loss
```

Prediction:
```
0.03
```

Bad prediction.
```
Large Loss
```

Skip-Gram typically uses:
```
Cross Entropy Loss
```
to measure prediction quality.


### Backpropagation
Prediction:
```
Wrong
```
↓
```
Loss
```
↓
```
Backpropagation
```
↓
```
Update Embeddings
```

#### Training Loop
```
Target Word
      ↓
Predict Context
      ↓
Loss
      ↓
Backpropagation
      ↓
Update Embeddings
```

Repeated millions or billions of times.


### Why Skip-Gram Learns Better Embeddings
CBOW:
```
Many Words
      ↓
One Prediction
```

Skip-Gram:
```
One Word
      ↓
Many Predictions
```

Result:
```
More Training Signal
```

for each word occurrence.

Therefore:
```
Better Embeddings
```

### Rare Word Advantage
Suppose corpus contains:
```
transformer
```

only a few times.

CBOW:
```
Few opportunities to learn.
```

Skip-Gram:

Each occurrence creates multiple prediction tasks.

Example:
```
transformer
```

appears once.

Window Size:
```
4
```

Creates:
```
4 Context Predictions
```

instead of:
```
1 Prediction
```

Result:
```
Rare Words Learn Faster
```

This is one reason Skip-Gram became very popular.

### How Semantic Similarity Emerges
Corpus:
```
The cat drinks milk

The dog drinks milk

The cat eats fish

The dog eats meat
```

Skip-Gram learns:
```
cat
```

predicts:
```
drinks

milk

eats
```

Similarly:
```
dog
```

predicts:
```
drinks

milk

eats
```

Therefore:
```
cat ≈ dog
```

Eventually:
```
cat = [0.82,0.14]

dog = [0.79,0.16]
```

Very similar vectors.

### Computational Challenge
Original Skip-Gram predicts:
```
Every Vocabulary Word
```

Vocabulary:
```
100,000 words
```

Softmax must compute:
```
100,000 probabilities
```

for every training step.

Very expensive.

This led to two important optimizations:
```
Negative Sampling

Hierarchical Softmax
```
These are among the most important Word2Vec optimizations.


### Advantages of Skip-Gram
#### Better Embeddings
Produces richer semantic representations.

#### Handles Rare Words Well
Each occurrence generates multiple training examples.

#### Strong Semantic Relationships
Captures similarity and analogies effectively.

#### Foundation of Modern Embeddings
Many later embedding methods build on Skip-Gram concepts.

### Limitations of Skip-Gram
#### Slower Training
Creates many prediction tasks.

#### Higher Computational Cost
More updates per word.

#### Static Embeddings
```
bank
```
always has one vector regardless of context.

#### Expensive Softmax
Large vocabularies make training costly.


### CBOW vs Skip-Gram
| Feature           | CBOW    | Skip-Gram |
| ----------------- | ------- | --------- |
| Input             | Context | Target    |
| Output            | Target  | Context   |
| Speed             | Faster  | Slower    |
| Rare Words        | Worse   | Better    |
| Embedding Quality | Good    | Better    |
| Training Data     | Smaller | Larger    |
| Computation       | Lower   | Higher    |
